# 01 — Embedding Space Explorer

## S01: Data Loading

In [ ]:
import asyncio
import struct
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import umap
from sklearn.decomposition import PCA

from amnesiac.store.db import get_connection
from amnesiac.store.embeddings import embed_query

In [ ]:
# allow imports from project root
sys.path.insert(0, str(Path.cwd().parent))

PERIODS = {
    "calm_2021":   ("2021-10-04", "2021-10-18"),
    "calm_2023":   ("2023-04-17", "2023-05-01"),
    "shock_covid": ("2020-04-06", "2020-04-20"),
    "shock_war":   ("2022-03-07", "2022-03-21"),
}
PERIOD = "calm_2021"  # change to switch period
PCA_N_COMPONENTS = 0.9  #128
TOP_K = 50           # documents per RAG query (placeholder for S03)
RAG_THRESHOLD = 0.5   # cosine similarity threshold for RAG relevance label
SECOND_PERIOD = "shock_war"  # second period for comparison
OUTLIER_QUANTILE = 1.0  # remove top 1% most distant points in PCA space
AXIS_CLIP_QUANTILE = 0.005  # clip axes to [q, 1-q] quantile range of umap_x/umap_y
SIM_THRESHOLDS = [0.3, 0.4, 0.5]  # thresholds to compare in retrieved/not-retrieved scatter

_EMBEDDING_DIM = 1536

In [ ]:
RAG_QUERIES = {
    "дкп": ["решение по ключевой ставке", "денежно-кредитная политика Банка России"],
    "инфляция": ["рост потребительских цен", "инфляция в России"],
    "продовольствие": ["цены на продукты питания"],
    "курс": ["курс рубля к доллару", "ослабление рубля"],
    "тарифы": ["повышение тарифов ЖКХ", "цены на бензин"],
    "зарплаты": ["рост зарплат", "реальные доходы населения"],
    "труд": ["дефицит кадров в России", "безработица в России"],
    "кризис": ["экономический кризис в России", "экономическая неопределённость", "санкции против России"],
    "бюджет": ["бюджетные расходы", "дефицит бюджета"],
}

In [ ]:
def load_period(conn, date_from: str, date_to: str) -> pd.DataFrame:
    """Load messages with embeddings for a given date range.

    JOIN chain: vec_messages.message_id -> processed_messages.id
                processed_messages.message_id -> messages.id
                messages.channel_id -> channels.id
    """
    sql = """
        SELECT
            vm.message_id   AS message_id,
            ch.username      AS channel,
            m.date           AS date,
            pm.processed_text AS processed_text,
            vm.embedding     AS embedding_blob
        FROM vec_messages vm
        JOIN processed_messages pm ON pm.id = vm.message_id
        JOIN messages m            ON m.id = pm.message_id
        JOIN channels ch           ON ch.id = m.channel_id
        WHERE m.date BETWEEN ? AND ?
          AND pm.is_valid = 1
        ORDER BY m.date
    """
    rows = conn.execute(sql, (date_from, date_to)).fetchall()

    records = []
    for message_id, channel, date_str, processed_text, emb_blob in rows:
        vec = np.array(
            struct.unpack(f"<{_EMBEDDING_DIM}f", emb_blob),
            dtype=np.float32,
        )
        records.append({
            "message_id": message_id,
            "channel": channel,
            "date": datetime.fromisoformat(date_str) if isinstance(date_str, str) else date_str,
            "processed_text": processed_text,
            "embedding": vec,
        })

    return pd.DataFrame(records, columns=["message_id", "channel", "date", "processed_text", "embedding"])

In [ ]:
async def _embed_queries(queries_dict: dict[str, list[str]]) -> dict[str, np.ndarray]:
    # flatten all query strings, embed concurrently
    flat_texts = []
    index_map: list[str] = []  # axis name for each text
    for axis, texts in queries_dict.items():
        for t in texts:
            flat_texts.append(t)
            index_map.append(axis)

    vectors = await asyncio.gather(*(embed_query(t) for t in flat_texts))

    # group vectors by axis and average
    from collections import defaultdict
    groups: dict[str, list] = defaultdict(list)
    for axis, vec in zip(index_map, vectors):
        groups[axis].append(vec)

    return {
        axis: np.mean(vecs, axis=0).astype(np.float32)
        for axis, vecs in groups.items()
    }


def embed_queries(queries_dict: dict[str, list[str]]) -> dict[str, np.ndarray]:
    """Embed all RAG query axes concurrently. Each axis -> mean of its query vectors."""
    return asyncio.run(_embed_queries(queries_dict))

In [ ]:
DB_PATH = Path("../data/db/blind_prophet.db")

conn = get_connection(str(DB_PATH))
date_from, date_to = PERIODS[PERIOD]
df = load_period(conn, date_from, date_to)
query_vectors = await _embed_queries(RAG_QUERIES)  # embed_queries(RAG_QUERIES)

In [ ]:
print(f"df.shape: {df.shape}")
print(f"\nMessages per channel:\n{df['channel'].value_counts().to_string()}")
if len(df) > 0:
    print(f"\nDate range: {df['date'].min()} — {df['date'].max()}")
else:
    print("\nNo messages loaded for this period.")
print(f"\nQuery vectors ({len(query_vectors)} axes):")
for axis, vec in query_vectors.items():
    print(f"  {axis}: shape {vec.shape}")

## S02: Geometry

In [ ]:
pio.templates.default = "plotly"  

def remove_outliers(
    df: pd.DataFrame,
    pca_coords: np.ndarray,
    quantile: float,
) -> tuple[pd.DataFrame, np.ndarray]:
    """Flag outliers in PCA space using Mahalanobis distance."""
    from numpy.linalg import pinv

    centroid = pca_coords.mean(axis=0)
    cov_inv = pinv(np.cov(pca_coords.T))
    diff = pca_coords - centroid
    distances = np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_inv, diff))

    threshold = np.quantile(distances, quantile)
    df = df.copy()
    df["is_outlier"] = distances > threshold
    return df, distances


def compute_umap(df: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """PCA(50) -> UMAP(2) projection. Adds umap_x, umap_y, is_outlier columns."""
    df = df.copy()
    X = np.stack(df["embedding"].values)
    # normalize to unit vectors
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    X = X / norms
    # store normalized embeddings back for downstream dot-product similarity
    df["embedding"] = list(X)

    X_pca = PCA(n_components=PCA_N_COMPONENTS, random_state=random_state).fit_transform(X)

    df, _ = remove_outliers(df, X_pca, OUTLIER_QUANTILE)
    inlier_mask = ~df["is_outlier"].values

    X_umap = umap.UMAP(
        n_components=2, random_state=random_state, metric="cosine"
    ).fit_transform(X_pca[inlier_mask])

    df["umap_x"] = np.nan
    df["umap_y"] = np.nan
    df.loc[inlier_mask, "umap_x"] = X_umap[:, 0]
    df.loc[inlier_mask, "umap_y"] = X_umap[:, 1]
    return df


def compute_rag_relevance(
    df: pd.DataFrame,
    query_vectors: dict[str, np.ndarray],
    threshold: float,
) -> pd.DataFrame:
    """Add `rag_relevant` bool column: max cosine sim to any query axis >= threshold."""
    df = df.copy()
    X = np.stack(df["embedding"].values)
    # ensure unit-norm (may already be normalized by compute_umap)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    X = X / norms

    # build query matrix (n_axes, 1536) and normalize
    Q = np.stack(list(query_vectors.values()))
    q_norms = np.linalg.norm(Q, axis=1, keepdims=True)
    q_norms[q_norms == 0] = 1.0
    Q = Q / q_norms

    # cosine similarity via dot product on unit vectors: (n_docs, n_axes)
    sims = X @ Q.T
    df["rag_relevant"] = sims.max(axis=1) >= threshold
    return df


def get_axis_limits(df, q=AXIS_CLIP_QUANTILE):
    return (
        df["umap_x"].quantile(q),   df["umap_x"].quantile(1 - q),
        df["umap_y"].quantile(q),   df["umap_y"].quantile(1 - q),
    )


In [ ]:
date_from, date_to = PERIODS[PERIOD]
df_calm = load_period(conn, date_from, date_to)

date_from2, date_to2 = PERIODS[SECOND_PERIOD]
df_shock = load_period(conn, date_from2, date_to2)

print(f"calm:  {df_calm.shape[0]} messages")
print(f"shock: {df_shock.shape[0]} messages")

In [ ]:
df_calm = compute_umap(df_calm)
df_calm = compute_rag_relevance(df_calm, query_vectors, RAG_THRESHOLD)

df_shock = compute_umap(df_shock)
df_shock = compute_rag_relevance(df_shock, query_vectors, RAG_THRESHOLD)

print(f"calm  — RAG relevant: {df_calm['rag_relevant'].sum()} / {len(df_calm)}")
print(f"shock — RAG relevant: {df_shock['rag_relevant'].sum()} / {len(df_shock)}")

for label, df_ in [(PERIOD, df_calm), (SECOND_PERIOD, df_shock)]:
    n_total = len(df_)
    n_out = df_["is_outlier"].sum()
    n_plot = n_total - n_out
    pct = n_out / n_total * 100
    print(f"{label}: {n_total} total, {n_out} outliers removed ({pct:.1f}%), {n_plot} plotted")


### Графики кластеризации

In [ ]:
df_calm["text_preview"] = df_calm["processed_text"].str[:120]
df_plot = df_calm[~df_calm["is_outlier"]]

fig1 = px.scatter(
    df_plot, x="umap_x", y="umap_y", color="channel",
    hover_data=["channel", "date", "text_preview"],
    title=f"{PERIOD} — by channel",
    opacity=0.7,
)
fig1.update_traces(marker_size=4)
fig1.update_layout(height=1000)
x0, x1, y0, y1 = get_axis_limits(df_plot)
fig1.update_xaxes(range=[x0, x1])
fig1.update_yaxes(range=[y0, y1])
fig1.show()


In [ ]:
df_plot = df_calm[~df_calm["is_outlier"]]

fig2 = px.scatter(
    df_plot, x="umap_x", y="umap_y", color="date",
    hover_data=["channel", "date", "text_preview"],
    title=f"{PERIOD} — by date",
    opacity=0.7,
)
fig2.update_traces(marker_size=4)
fig2.update_layout(height=1000)
x0, x1, y0, y1 = get_axis_limits(df_plot)
fig2.update_xaxes(range=[x0, x1])
fig2.update_yaxes(range=[y0, y1])
fig2.show()


In [ ]:
df_plot = df_calm[~df_calm["is_outlier"]]

fig3 = px.scatter(
    df_plot, x="umap_x", y="umap_y", color="rag_relevant",
    hover_data=["channel", "date", "text_preview"],
    title=f"{PERIOD} — RAG relevance (threshold={RAG_THRESHOLD})",
    opacity=0.7,
)
fig3.update_traces(marker_size=4)
fig3.update_layout(height=1000)
x0, x1, y0, y1 = get_axis_limits(df_plot)
fig3.update_xaxes(range=[x0, x1])
fig3.update_yaxes(range=[y0, y1])
fig3.show()


In [ ]:
df_shock["text_preview"] = df_shock["processed_text"].str[:120]
df_plot = df_shock[~df_shock["is_outlier"]]

fig4 = px.scatter(
    df_plot, x="umap_x", y="umap_y", color="channel",
    hover_data=["channel", "date", "text_preview"],
    title=f"{SECOND_PERIOD} — by channel",
    opacity=0.7,
)
fig4.update_traces(marker_size=4)
fig4.update_layout(height=1000)
x0, x1, y0, y1 = get_axis_limits(df_plot)
fig4.update_xaxes(range=[x0, x1])
fig4.update_yaxes(range=[y0, y1])
fig4.show()


In [ ]:
df_plot = df_shock[~df_shock["is_outlier"]]

fig5 = px.scatter(
    df_plot, x="umap_x", y="umap_y", color="date",
    hover_data=["channel", "date", "text_preview"],
    title=f"{SECOND_PERIOD} — by date",
    opacity=0.7,
)
fig5.update_traces(marker_size=4)
fig5.update_layout(height=1000)
x0, x1, y0, y1 = get_axis_limits(df_plot)
fig5.update_xaxes(range=[x0, x1])
fig5.update_yaxes(range=[y0, y1])
fig5.show()


In [ ]:
df_plot = df_shock[~df_shock["is_outlier"]]

fig6 = px.scatter(
    df_plot, x="umap_x", y="umap_y", color="rag_relevant",
    hover_data=["channel", "date", "text_preview"],
    title=f"{SECOND_PERIOD} — RAG relevance (threshold={RAG_THRESHOLD})",
    opacity=0.7,
)
fig6.update_traces(marker_size=4)
fig6.update_layout(height=800)
x0, x1, y0, y1 = get_axis_limits(df_plot)
fig6.update_xaxes(range=[x0, x1])
fig6.update_yaxes(range=[y0, y1])
fig6.show()


In [ ]:
# Combined UMAP: both periods projected together
df_calm_c = df_calm.copy()
df_calm_c["period"] = PERIOD
df_shock_c = df_shock.copy()
df_shock_c["period"] = SECOND_PERIOD

df_combined = pd.concat([df_calm_c, df_shock_c], ignore_index=True)

# re-run PCA -> UMAP on the union
X_all = np.stack(df_combined["embedding"].values)
norms = np.linalg.norm(X_all, axis=1, keepdims=True)
norms[norms == 0] = 1.0
X_all = X_all / norms

X_pca = PCA(n_components=PCA_N_COMPONENTS, random_state=42).fit_transform(X_all)  # PCA_N_COMPONENTS

In [ ]:
X_pca.shape

In [ ]:
X_all.shape

In [ ]:
df_combined, _ = remove_outliers(df_combined, X_pca, OUTLIER_QUANTILE)
inlier_mask = ~df_combined["is_outlier"].values

X_umap = umap.UMAP(n_components=2, random_state=42, metric="cosine").fit_transform(X_pca[inlier_mask])

df_combined["umap_x"] = np.nan
df_combined["umap_y"] = np.nan
df_combined.loc[inlier_mask, "umap_x"] = X_umap[:, 0]
df_combined.loc[inlier_mask, "umap_y"] = X_umap[:, 1]
df_combined["text_preview"] = df_combined["processed_text"].str[:120]

n_total = len(df_combined)
n_out = df_combined["is_outlier"].sum()
n_plot = n_total - n_out
pct = n_out / n_total * 100
print(f"combined: {n_total} total, {n_out} outliers removed ({pct:.1f}%), {n_plot} plotted")

df_plot = df_combined[~df_combined["is_outlier"]]

fig7 = px.scatter(
    df_plot, x="umap_x", y="umap_y", color="period",
    hover_data=["channel", "date", "text_preview"],
    title=f"{PERIOD} vs {SECOND_PERIOD}",
    opacity=0.7,
)
fig7.update_traces(marker_size=4)
fig7.update_layout(height=1000)
x0, x1, y0, y1 = get_axis_limits(df_plot)
fig7.update_xaxes(range=[x0, x1])
fig7.update_yaxes(range=[y0, y1])
fig7.show()

# S02: Выводы — геометрия пространства эмбеддингов

## calm_2021 — by channel

Основное облако (~7500 сообщений) содержит все 9 каналов в равномерном перемешении без выраженной источниковой сегрегации. **Source bias в общем корпусе отсутствует** — каналы освещают схожую повестку в схожем семантическом пространстве.

Исключение — **prime1** (зелёный): канал образует обособленную зону в нижней части графика с характерными линейными цепочками плотно прилегающих точек. Это структурный артефакт жанра — шаблонные маркет-брифы с повторяющейся структурой текста. Семантически они близки между собой, но удалены от остального корпуса. Confirmed known limitation из E03.

## calm_2021 — by date

Цвета равномерно перемешаны по всему облаку без видимого дрейфа от начала к концу периода. **Темпоральный дрейфт внутри двухнедельного окна отсутствует.** Новостная повестка за 14 дней остаётся тематически стабильной — это подтверждает корректность двухнедельного окна как единицы выборки.

## calm_2021 — RAG relevance (threshold=0.5)

При пороге 0.5 релевантных точек критически мало, и они концентрируются преимущественно в зоне prime1 — то есть запросы на уровне 0.5 захватывают шаблонные брифы, но не аналитический новостной текст. **Порог 0.5 для спокойного периода завышен.** Экономическая повестка октября 2021 не образует высококонтрастного пика в пространстве FRIDA — сигнал размыт по облаку. Подбор реального порога — задача S03.

## shock_war — by channel

Геометрия принципиально иная: облако вытянуто и структурировано, prime1-цепочки ещё более выражены и образуют отдельные полуострова в левой части. Остальные каналы в основном облаке по-прежнему перемешаны. **Шоковый период не нарушает межканальный баланс** в основной массе текстов — источниковой сегрегации в зоне общей повестки нет.

## shock_war — by date

Цвета перемешаны равномерно, темпоральный дрейфт не выражен. Шок марта 2022 не привёл к постепенному тематическому сдвигу внутри двухнедельного окна — повестка переключилась резко и оставалась стабильной на всём периоде.

## shock_war — RAG relevance (threshold=0.5)

Красных точек заметно больше чем в calm_2021, но они снова концентрируются в prime1-зоне. Парадокс: запросы про санкции, курс и кризис при пороге 0.5 лучше захватывают шаблонные финансовые брифы, чем аналитические тексты основных каналов. Возможное объяснение — брифы содержат числовые котировки и названия инструментов, которые FRIDA кодирует ближе к экономическим запросам, чем нарративный текст.

## calm_2021 vs shock_war — совмещённый

Основные облака обоих периодов **хорошо перекрываются** в общем пространстве — общая тематическая структура сохраняется между периодами. Уникальная зона shock_war — левая часть с prime1-цепочками, которых в октябре 2021 практически нет. Разрастание prime1-зоны в шоковый период объясняется ростом объёма финансовых сводок в условиях волатильности.

---

## Общие выводы

**Нет source bias** в основном корпусе: ни в спокойный, ни в шоковый период каналы не занимают обособленные тематические ниши. RAG-запросы по теме получат контрибуцию от всех каналов пропорционально их объёму.

**Нет темпорального дрейфта** внутри двухнедельных окон: двухнедельный период — корректная единица выборки, агрегация данных за окно не смешивает разные тематические состояния.

**Prime1 — структурный выброс**, не near-duplicate с другими каналами. Это отдельный жанр (маркет-брифы), семантически отличный от новостного нарратива. Требует отдельного решения перед суммаризацией: либо фильтрация по источнику, либо явная метка жанра.

**RAG_THRESHOLD = 0.5 завышен для спокойных периодов.** Порог работает лишь для шоковых периодов с высококонтрастной повесткой, и даже там захватывает преимущественно шаблонный контент. Реальный рабочий порог предположительно в диапазоне 0.3–0.4 — уточняется в S03 по распределению similarity scores.

**Следующий шаг (S03):** распределение similarity scores по осям, Jaccard-матрица перекрытия запросов, подбор порога.

## S03: RAG Quality & Query Overlap (TODO)

In [ ]:
def compute_similarity_matrix(
    df: pd.DataFrame,
    query_vectors: dict[str, np.ndarray],
) -> pd.DataFrame:
    """Add sim_<axis> and max_sim columns to df using dot product on unit-normalized embeddings."""
    df = df.copy()
    X = np.stack(df["embedding"].values)  # (n, 1536), already unit-normalized from compute_umap

    axes = list(query_vectors.keys())
    Q = np.stack([query_vectors[a] for a in axes])  # (9, 1536)
    # normalize query vectors
    q_norms = np.linalg.norm(Q, axis=1, keepdims=True)
    q_norms[q_norms == 0] = 1.0
    Q = Q / q_norms

    sims = X @ Q.T  # (n, 9)
    for i, axis in enumerate(axes):
        df[f"sim_{axis}"] = sims[:, i]
    df["max_sim"] = sims.max(axis=1)
    return df

In [ ]:
df_calm = compute_similarity_matrix(df_calm, query_vectors)
df_shock = compute_similarity_matrix(df_shock, query_vectors)

print(f"calm  max_sim: mean={df_calm['max_sim'].mean():.3f}, median={df_calm['max_sim'].median():.3f}")
print(f"shock max_sim: mean={df_shock['max_sim'].mean():.3f}, median={df_shock['max_sim'].median():.3f}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Figure 1 — max_sim histogram: calm vs shock overlaid
fig_hist = go.Figure()
fig_hist.add_trace(go.Histogram(
    x=df_calm["max_sim"], nbinsx=50, name=PERIOD,
    opacity=0.6, xbins=dict(start=0, end=1, size=1/50),
))
fig_hist.add_trace(go.Histogram(
    x=df_shock["max_sim"], nbinsx=50, name=SECOND_PERIOD,
    opacity=0.6, xbins=dict(start=0, end=1, size=1/50),
))
for t in SIM_THRESHOLDS:
    fig_hist.add_vline(x=t, line_dash="dash", line_color="gray",
                       annotation_text=str(t), annotation_position="top right")
fig_hist.update_layout(
    barmode="overlay",
    title="Max similarity to any RAG axis — calm_2021 vs shock_war",
    xaxis_title="max_sim",
    yaxis_title="count",
    xaxis=dict(range=[0, 1]),
)
fig_hist.show()

In [ ]:
# Figure 2 — per-axis similarity histograms (3×3 grid)
axes = list(query_vectors.keys())
fig_axes = make_subplots(rows=3, cols=3, subplot_titles=axes)

for idx, axis in enumerate(axes):
    row = idx // 3 + 1
    col = idx % 3 + 1
    col_name = f"sim_{axis}"
    fig_axes.add_trace(
        go.Histogram(x=df_calm[col_name], nbinsx=40, name=PERIOD,
                     opacity=0.6, showlegend=(idx == 0),
                     xbins=dict(start=0, end=1, size=1/40)),
        row=row, col=col,
    )
    fig_axes.add_trace(
        go.Histogram(x=df_shock[col_name], nbinsx=40, name=SECOND_PERIOD,
                     opacity=0.6, showlegend=(idx == 0),
                     xbins=dict(start=0, end=1, size=1/40)),
        row=row, col=col,
    )

fig_axes.update_layout(
    barmode="overlay",
    title_text="Similarity distribution by axis",
    height=800,
)
fig_axes.show()

In [ ]:
# Figure 3 — retrieved/not-retrieved scatter in UMAP space (df_calm inliers only)
df_calm_inliers = df_calm[~df_calm["is_outlier"]].copy()
x0, x1, y0, y1 = get_axis_limits(df_calm_inliers)

fig_scatter = make_subplots(rows=1, cols=len(SIM_THRESHOLDS),
                             subplot_titles=[f"threshold={t}" for t in SIM_THRESHOLDS])

for col_idx, t in enumerate(SIM_THRESHOLDS, start=1):
    retrieved = df_calm_inliers["max_sim"] >= t
    for label, mask, color in [("retrieved", retrieved, "#2166ac"),
                                ("not retrieved", ~retrieved, "#d1d1d1")]:
        subset = df_calm_inliers[mask]
        fig_scatter.add_trace(
            go.Scatter(
                x=subset["umap_x"], y=subset["umap_y"],
                mode="markers",
                marker=dict(size=3, color=color, opacity=0.6),
                name=label,
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )
    fig_scatter.update_xaxes(range=[x0, x1], row=1, col=col_idx)
    fig_scatter.update_yaxes(range=[y0, y1], row=1, col=col_idx)

fig_scatter.update_layout(
    title_text=f"{PERIOD} — retrieved vs not retrieved by threshold",
    height=500,
)
fig_scatter.show()

In [ ]:
def compute_jaccard(
    df: pd.DataFrame,
    query_vectors: dict[str, np.ndarray],
    threshold: float,
) -> pd.DataFrame:
    """9×9 Jaccard similarity matrix for retrieved sets at a given threshold."""
    axes = list(query_vectors.keys())
    sets = {axis: set(df.loc[df[f"sim_{axis}"] >= threshold, "message_id"]) for axis in axes}
    n = len(axes)
    matrix = np.zeros((n, n))
    for i, a in enumerate(axes):
        for j, b in enumerate(axes):
            union = sets[a] | sets[b]
            matrix[i, j] = len(sets[a] & sets[b]) / len(union) if union else 0.0
    return pd.DataFrame(matrix, index=axes, columns=axes)

In [ ]:
# Figure 4 — Jaccard heatmaps: calm | shock side by side
jac_calm = compute_jaccard(df_calm, query_vectors, RAG_THRESHOLD)
jac_shock = compute_jaccard(df_shock, query_vectors, RAG_THRESHOLD)
axes = list(query_vectors.keys())

fig_jac = make_subplots(rows=1, cols=2,
                         subplot_titles=[f"{PERIOD}", f"{SECOND_PERIOD}"])

for col_idx, (label, jac) in enumerate([(PERIOD, jac_calm), (SECOND_PERIOD, jac_shock)], start=1):
    z = jac.values
    text = [[f"{v:.2f}" for v in row] for row in z]
    fig_jac.add_trace(
        go.Heatmap(
            z=z, x=axes, y=axes,
            text=text, texttemplate="%{text}",
            colorscale="Blues", zmin=0, zmax=1,
            showscale=(col_idx == 2),
        ),
        row=1, col=col_idx,
    )

fig_jac.update_layout(
    title_text=f"Jaccard overlap between axes (threshold={RAG_THRESHOLD})",
    height=500,
)
fig_jac.show()

# S03: Выводы — качество RAG-запросов

## Max-similarity histogram

Оба периода дают унимодальное распределение с пиком около 0.25–0.30 без выраженного провала между «сигналом» и «шумом». **Естественной границы для threshold-based отбора не существует** — любой абсолютный порог режет по живому, а не по смысловой границе.

Разница между периодами невелика: медиана calm_2021 = 0.301, shock_war = 0.328. Шоковый период даёт более тяжёлый правый хвост — часть документов действительно высококонтрастна по теме, но основная масса остаётся в той же зоне.

## Similarity по осям

**Шок-специфичные оси** (shock_war заметно правее calm_2021): курс, ДКП. В феврале 2022 эти темы доминировали — сигнал чистый.

**Симметричные оси** (периоды похожи): инфляция, кризис, бюджет. Ось инфляции работает стабильно в обоих периодах — это ценное свойство для нашей задачи прогноза.

**Calm-специфичная ось**: продовольствие — чуть выше в октябре 2021, что соответствует реальной повестке того периода.

**Слабые оси**: зарплаты, труд — низкий signal-to-noise в обоих периодах, пик сдвинут влево.

## UMAP scatter по порогам (calm_2021)

При любом пороге retrieved-документы концентрируются преимущественно в prime1-зоне, а не в основном новостном облаке. Threshold-based отбор захватывает шаблонный контент эффективнее, чем нарративный — артефакт жанровой структуры prime1.

## Jaccard-матрица

При threshold=0.5 в calm_2021 — нули везде: документов выше порога почти нет, матрица вырождена.

В shock_war значимые пересечения:
- **зарплаты / труд: 0.25** — ожидаемо, тематически близкие оси
- **инфляция / курс: 0.23** — в шоковый период темпы роста цен и ослабление рубля освещаются в одном контексте
- **ДКП / курс: 0.16** — умеренное перекрытие, оправданное

Остальные пары независимы.

---

## Решения

**Стратегия отбора: top-K вместо threshold.** Абсолютный порог similarity не работает для данного корпуса — унимодальное распределение без естественной границы делает его ненадёжным. Переходим на top-K по каждой оси: фиксированное число ближайших документов на запрос вне зависимости от абсолютного значения similarity.

**K = 50 на ось** как стартовое значение (итого до 450 документов до дедупликации). Уточняется по результатам суммаризации в E05.

**Оси зарплаты + труд**: кандидаты на объединение в одну ось при top-K — проверить в E05 насколько они дают разный контент при K=50.

**Prime1**: исключить из RAG-выборки на этапе E05. Шаблонные брифы засоряют top-K по всем осям и не несут нарративного сигнала нужного для суммаризации. Реализация: фильтр `channel != 'prime1'` перед векторным поиском.

## S03 Findings
- Recommended threshold: (fill after running)
- High-overlap axis pairs: (fill after running)
- Axes with weak signal in calm period: (fill after running)

## S04: Near-Duplicates (TODO)

In [ ]:
NEAR_DUP_MIN_SIM = 0.7   # minimum similarity to consider as candidate pair
NEAR_DUP_THRESHOLD = 0.9  # threshold for near-duplicate classification

In [ ]:
def compute_pairs(df: pd.DataFrame, min_sim: float) -> pd.DataFrame:
    """Compute all pairwise similarities among inliers and return pairs above min_sim."""
    inliers = df[~df["is_outlier"]].reset_index(drop=True)
    embeddings = np.stack(inliers["embedding"].values)  # already unit-normalized

    sim_matrix = embeddings @ embeddings.T
    np.fill_diagonal(sim_matrix, 0.0)

    n = len(inliers)
    rows, cols = np.triu_indices(n, k=1)
    sims = sim_matrix[rows, cols]

    mask = sims >= min_sim
    rows, cols, sims = rows[mask], cols[mask], sims[mask]

    pairs = pd.DataFrame({
        "msg_id_a":   inliers["message_id"].values[rows],
        "msg_id_b":   inliers["message_id"].values[cols],
        "channel_a":  inliers["channel"].values[rows],
        "channel_b":  inliers["channel"].values[cols],
        "date_a":     inliers["date"].values[rows],
        "date_b":     inliers["date"].values[cols],
        "text_a":     inliers["processed_text"].values[rows],
        "text_b":     inliers["processed_text"].values[cols],
        "similarity": sims,
    })

    pairs["cross_channel"] = pairs["channel_a"] != pairs["channel_b"]
    pairs["time_diff_minutes"] = (
        (pairs["date_a"] - pairs["date_b"]).abs() / pd.Timedelta(minutes=1)
    )

    return pairs.reset_index(drop=True)

In [ ]:
pairs_calm = compute_pairs(df_calm, NEAR_DUP_MIN_SIM)
pairs_shock = compute_pairs(df_shock, NEAR_DUP_MIN_SIM)
print(f"calm_2021:  {len(pairs_calm)} pairs above {NEAR_DUP_MIN_SIM}, {pairs_calm['cross_channel'].sum()} cross-channel")
print(f"shock_war: {len(pairs_shock)} pairs above {NEAR_DUP_MIN_SIM}, {pairs_shock['cross_channel'].sum()} cross-channel")

In [ ]:
# Figure 1 — similarity histogram for cross-channel pairs above min_sim
cross_calm = pairs_calm[pairs_calm["cross_channel"]]
cross_shock = pairs_shock[pairs_shock["cross_channel"]]

fig_s04_1 = go.Figure()
fig_s04_1.add_trace(go.Histogram(
    x=cross_calm["similarity"], nbinsx=30, name=PERIOD,
    opacity=0.6,
    xbins=dict(start=NEAR_DUP_MIN_SIM, end=1.0, size=(1.0 - NEAR_DUP_MIN_SIM) / 30),
))
fig_s04_1.add_trace(go.Histogram(
    x=cross_shock["similarity"], nbinsx=30, name=SECOND_PERIOD,
    opacity=0.6,
    xbins=dict(start=NEAR_DUP_MIN_SIM, end=1.0, size=(1.0 - NEAR_DUP_MIN_SIM) / 30),
))
fig_s04_1.add_vline(
    x=NEAR_DUP_THRESHOLD, line_dash="dash", line_color="red",
    annotation_text=f"threshold={NEAR_DUP_THRESHOLD}",
    annotation_position="top right",
)
fig_s04_1.update_layout(
    barmode="overlay",
    title=f"Pairwise similarity distribution (pairs > {NEAR_DUP_MIN_SIM})",
    xaxis_title="similarity",
    yaxis_title="count",
    xaxis=dict(range=[NEAR_DUP_MIN_SIM, 1.0]),
)
fig_s04_1.show()

In [ ]:
# Figure 2 — channel pair heatmap for cross-channel near-duplicates above threshold
def _channel_pair_matrix(pairs: pd.DataFrame, threshold: float) -> pd.DataFrame:
    above = pairs[(pairs["cross_channel"]) & (pairs["similarity"] >= threshold)].copy()
    counts = above.groupby(["channel_a", "channel_b"]).size().reset_index(name="count")
    all_channels = sorted(set(pairs["channel_a"]) | set(pairs["channel_b"]))
    matrix = pd.DataFrame(0, index=all_channels, columns=all_channels)
    for _, row in counts.iterrows():
        matrix.loc[row["channel_a"], row["channel_b"]] += row["count"]
        matrix.loc[row["channel_b"], row["channel_a"]] += row["count"]
    return matrix


mat_calm = _channel_pair_matrix(pairs_calm, NEAR_DUP_THRESHOLD)
mat_shock = _channel_pair_matrix(pairs_shock, NEAR_DUP_THRESHOLD)

# align columns/index to union of channels
all_ch = sorted(set(mat_calm.columns) | set(mat_shock.columns))
mat_calm = mat_calm.reindex(index=all_ch, columns=all_ch, fill_value=0)
mat_shock = mat_shock.reindex(index=all_ch, columns=all_ch, fill_value=0)

fig_s04_2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"calm ({PERIOD})", f"shock ({SECOND_PERIOD})"],
)
for col_idx, (label, mat) in enumerate([(PERIOD, mat_calm), (SECOND_PERIOD, mat_shock)], start=1):
    z = mat.values.tolist()
    text = [[str(v) for v in row] for row in mat.values]
    fig_s04_2.add_trace(
        go.Heatmap(
            z=z, x=all_ch, y=all_ch,
            text=text, texttemplate="%{text}",
            colorscale="Blues",
            showscale=(col_idx == 2),
        ),
        row=1, col=col_idx,
    )

fig_s04_2.update_layout(
    title_text=f"Cross-channel near-duplicate pairs (threshold={NEAR_DUP_THRESHOLD})",
    height=500,
)
fig_s04_2.show()

In [ ]:
# Examples table — top 10 most similar cross-channel pairs above threshold
_display_cols = ["similarity", "channel_a", "channel_b", "time_diff_minutes", "text_a", "text_b"]

def _top10_examples(pairs: pd.DataFrame, threshold: float) -> pd.DataFrame:
    above = pairs[(pairs["cross_channel"]) & (pairs["similarity"] >= threshold)]
    top = above.nlargest(10, "similarity")[_display_cols].copy()
    top["text_a"] = top["text_a"].str[:100]
    top["text_b"] = top["text_b"].str[:100]
    return top.reset_index(drop=True)

print(f"=== {PERIOD} — top 10 cross-channel near-duplicates (sim >= {NEAR_DUP_THRESHOLD}) ===")
display(_top10_examples(pairs_calm, NEAR_DUP_THRESHOLD))

print(f"\n=== {SECOND_PERIOD} — top 10 cross-channel near-duplicates (sim >= {NEAR_DUP_THRESHOLD}) ===")
display(_top10_examples(pairs_shock, NEAR_DUP_THRESHOLD))

In [ ]:
# Figure 3 — time_diff_minutes histogram for cross-channel near-duplicates
near_dups_calm = pairs_calm[(pairs_calm["cross_channel"]) & (pairs_calm["similarity"] >= NEAR_DUP_THRESHOLD)]
near_dups_shock = pairs_shock[(pairs_shock["cross_channel"]) & (pairs_shock["similarity"] >= NEAR_DUP_THRESHOLD)]

fig_s04_3 = go.Figure()
fig_s04_3.add_trace(go.Histogram(
    x=near_dups_calm["time_diff_minutes"], nbinsx=24, name=PERIOD,
    opacity=0.6,
    xbins=dict(start=0, end=120, size=5),
))
fig_s04_3.add_trace(go.Histogram(
    x=near_dups_shock["time_diff_minutes"], nbinsx=24, name=SECOND_PERIOD,
    opacity=0.6,
    xbins=dict(start=0, end=120, size=5),
))
fig_s04_3.update_layout(
    barmode="overlay",
    title="Time difference between cross-channel near-duplicates (minutes)",
    xaxis_title="time_diff_minutes",
    yaxis_title="count",
    xaxis=dict(range=[0, 120]),
)
fig_s04_3.show()

# S04: Выводы — near-duplicates

## Масштаб проблемы

За двухнедельный период кросс-канальных пар выше similarity 0.7:
- calm_2021: 103 787 пар
- shock_war: 306 314 пар (почти втрое больше — шоковый период генерирует взрывной поток wire-новостей)

Выше порога 0.9 (near-duplicates в строгом смысле) пар на порядок меньше — хвост гистограммы практически обнуляется.

## Гистограмма similarity

Экспоненциальное убывание от 0.7 до 0.9, за порогом 0.9 — хвост близкий к нулю. Никакого бимодального распределения — граница near-duplicate не самоочевидна из формы распределения, но порог 0.9 находится в зоне естественного хвоста и соответствует визуально проверенным примерам.

## Основные канальные пары

**calm_2021:** ТАСС / РИА (122), ТАСС / РБК (62), РИА / РБК (50). Классическая тройка федеральных wire-агентств, синхронно публикующих одни и те же сообщения.

**shock_war:** prime1 / ifax_go (616) — доминирует с отрывом. Это два канала одной группы (Интерфакс), синхронно публикующие маркет-брифы. ТАСС / РБК (296), ТАСС / РИА (271) — wire-дубли в условиях высокой новостной активности.

## Временно́й паттерн

Пик time_diff в первых 0–5 минутах, резкое экспоненциальное убывание. Подавляющее большинство near-duplicates — это синхронная публикация одного wire-сообщения несколькими каналами в течение нескольких минут. Хвост до 120 минут — редакционная переработка с задержкой или случайно похожие тексты.

## Примеры

Строки с similarity 0.999 — буквально идентичный текст, разница публикации 1–11 минут (ТАСС → Фонтанка, ТАСС → РИА). Строка с similarity 0.992 — одна и та же новость с минорной редакционной правкой («с 10.00 мск» vs «с 10:00 мск»).

---

## Решения

**Рекомендуемый порог дедупликации: 0.9.** Выше этого значения только реальные wire-дубли с временны́м окном в единицы минут. Порог чистый, ложных срабатываний на проверенных примерах нет.

**Стратегия дедупликации:** при формировании top-K выборки для суммаризации — после RAG-отбора применять дедупликацию: из группы near-duplicate пар (similarity ≥ 0.9) оставлять один документ (например, с наибольшим числом символов или первый по времени). Реализация в E05.

**Prime1 / ifax_go:** пара 616 дублей в shock_war — следствие архитектуры Интерфакс-группы. При исключении prime1 из RAG-выборки (решение S03) ifax_go остаётся, но его объём невелик — дополнительной фильтрации не требуется.

**Дедупликация не заменяет фильтр prime1:** prime1 — жанровый выброс, его исключение обосновано независимо от near-duplicate логики.

In [ ]:
SANITY_K = 50          # top-K documents per axis
SANITY_AXES = ["инфляция", "дкп", "курс", "продовольствие"]  # axes to inspect manually
TOKENS_PER_CHAR = 1 / 3.5  # rough estimate for Russian text

In [ ]:
def top_k_by_axis(df, query_vectors, k, exclude_channel=None):
    """Return dict[axis -> DataFrame] with top-k documents per axis."""
    d = df.copy()
    if exclude_channel:
        d = d[d["channel"] != exclude_channel]
    d = d[~d["is_outlier"]].reset_index(drop=True)

    result = {}
    for axis in query_vectors:
        col = f"sim_{axis}"
        if col not in d.columns:
            # recompute if needed
            q = query_vectors[axis]
            q = q / np.linalg.norm(q)
            X = np.stack(d["embedding"].values)
            d[col] = X @ q
        top = d.nlargest(k, col)[["message_id", "channel", "date", "processed_text", col]].copy()
        top = top.rename(columns={col: "similarity"})
        result[axis] = top
    return result


topk_calm  = top_k_by_axis(df_calm,  query_vectors, SANITY_K, exclude_channel="prime1")
topk_shock = top_k_by_axis(df_shock, query_vectors, SANITY_K, exclude_channel="prime1")

In [ ]:
def overlap_stats(topk: dict[str, pd.DataFrame], period_name: str):
    sets = {axis: set(df["message_id"]) for axis, df in topk.items()}
    union_ids = set().union(*sets.values())

    # count how many axes each document appears in
    id_counts = {}
    for axis_set in sets.values():
        for mid in axis_set:
            id_counts[mid] = id_counts.get(mid, 0) + 1

    counts = pd.Series(id_counts.values()).value_counts().sort_index()

    print(f"\n{period_name}")
    print(f"  9 axes × {SANITY_K} = {9 * SANITY_K} slots")
    print(f"  union unique docs: {len(union_ids)}")
    print(f"  docs by axis count:")
    for n_axes, n_docs in counts.items():
        print(f"    in {n_axes} axis/axes: {n_docs} docs")


overlap_stats(topk_calm,  "calm_2021")
overlap_stats(topk_shock, "shock_war")

In [ ]:
import plotly.express as px

def temporal_topk(topk: dict, period_name: str):
    rows = []
    for axis, df in topk.items():
        for _, row in df.iterrows():
            rows.append({"axis": axis, "date": pd.Timestamp(row["date"]).date()})
    tdf = pd.DataFrame(rows)
    counts = tdf.groupby(["date", "axis"]).size().reset_index(name="count")

    fig = px.bar(
        counts, x="date", y="count", color="axis", barmode="stack",
        title=f"Top-{SANITY_K} documents per axis by day — {period_name}",
        height=400,
    )
    fig.show()


temporal_topk(topk_calm,  "calm_2021")
temporal_topk(topk_shock, "shock_war")

In [ ]:
def token_budget(topk: dict, period_name: str):
    union_texts = {}
    for axis_df in topk.values():
        for _, row in axis_df.iterrows():
            union_texts[row["message_id"]] = row["processed_text"]

    lengths = [len(t) for t in union_texts.values()]
    total_chars = sum(lengths)
    total_tokens = int(total_chars * TOKENS_PER_CHAR)

    print(f"\n{period_name}")
    print(f"  unique docs in union: {len(union_texts)}")
    print(f"  avg doc length: {int(total_chars / len(lengths))} chars / ~{int(total_chars / len(lengths) * TOKENS_PER_CHAR)} tokens")
    print(f"  total: {total_chars:,} chars / ~{total_tokens:,} tokens")


token_budget(topk_calm,  "calm_2021")
token_budget(topk_shock, "shock_war")

In [ ]:
def show_top(topk: dict, axis: str, period_name: str, n=10):
    df = topk[axis].head(n)
    print(f"\n{'='*60}")
    print(f"Top-{n} | axis={axis} | {period_name}")
    print(f"{'='*60}")
    for _, row in df.iterrows():
        print(f"[{row['channel']}] sim={row['similarity']:.3f} | {str(row['date'])[:10]}")
        print(row["processed_text"][:300])
        print()


for axis in SANITY_AXES:
    show_top(topk_calm,  axis, "calm_2021")
    show_top(topk_shock, axis, "shock_war")

In [ ]:
# Per-axis temporal distribution — кто концентрируется, кто размазан
def axis_temporal_df(topk, period):
    frames = []
    for axis, d in topk.items():
        g = (d.assign(day=pd.to_datetime(d["date"]).dt.date)
               .groupby("day").size().reset_index(name="count"))
        g["axis"] = axis
        g["period"] = period
        frames.append(g)
    return pd.concat(frames, ignore_index=True)

for name, topk in [(PERIOD, topk_calm), (SECOND_PERIOD, topk_shock)]:
    df_t = axis_temporal_df(topk, name)
    fig = px.bar(
        df_t, x="day", y="count",
        facet_col="axis", facet_col_wrap=3,
        title=f"{name} — top-{SANITY_K} docs per axis by day",
        height=700,
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_xaxes(tickformat="%m-%d")
    fig.update_yaxes(matches=None)  # каждая ось со своей y-шкалой
    fig.show()

In [ ]:
# Token budget — union top-K across axes, deduped by message_id
def token_budget(topk, period):
    all_docs = pd.concat(topk.values(), ignore_index=True)
    unique = all_docs.drop_duplicates("message_id")
    chars = unique["processed_text"].str.len()
    total_chars = chars.sum()
    total_tokens = int(total_chars * TOKENS_PER_CHAR)
    print(f"{period}: {len(unique)} unique docs | "
          f"avg {chars.mean():.0f} ch, median {chars.median():.0f} ch, max {chars.max()} ch | "
          f"total {total_chars:,} ch ≈ {total_tokens:,} tokens")

token_budget(topk_calm,  PERIOD)
token_budget(topk_shock, SECOND_PERIOD)